# Class 3. Measurement, Observables, and the Quantum Toolbox (Exercises)

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Compute and interpret expectation values for single- and two-qubit states.
2. Understand shot-based estimation and confidence bounds.
3. Use Qiskit primitives (`StatevectorSampler`, `StatevectorEstimator`) correctly.
4. Connect quantum measurement features to a real-data classification pipeline.

**Convention reminder.** Unless stated otherwise, each qubit starts in $\lvert 0\rangle$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram

import pennylane as qml

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

---
## Background: observables, eigenpairs, Born rule

An **observable** is a Hermitian operator $O$.

- It has eigenpairs $(\lambda_i,\lvert\phi_i\rangle)$ and a spectral decomposition
  $O = \sum_i \lambda_i\lvert\phi_i\rangle\langle\phi_i\rvert$.
- A projective measurement of $O$ returns an eigenvalue $\lambda_i$ and leaves the system in the corresponding eigenstate $\lvert\phi_i\rangle$.

If the system is in $\lvert\psi\rangle$, the **Born rule** gives
\[
P(\lambda_i)=\lvert\langle\phi_i\vert\psi\rangle\rvert^2.
\]
For real unit vectors, $\lvert\langle\phi_i\vert\psi\rangle\rvert=\cos\theta$, hence $P=\cos^2\theta$.

Convention: $\langle\psi\vert=(\lvert\psi\rangle)^\dagger$ (conjugate transpose). In NumPy, `np.vdot(a, b)` computes $a^\dagger b$.

The **expectation value** is the mean outcome:
\[
\langle O\rangle = \langle\psi\vert O\vert\psi\rangle = \sum_i \lambda_i P(\lambda_i).
\]

### M3.0. Born rule as overlap; expectation as weighted average

For
$\lvert\psi\rangle = \cos(\pi/8)\lvert0\rangle + \sin(\pi/8)\lvert1\rangle$,
compute $P(0)$ and $P(1)$ via overlaps with $\lvert0\rangle,\lvert1\rangle$. Then compute $\langle Z\rangle$ as a weighted average of eigenvalues $\{+1,-1\}$ and verify it matches $\langle\psi\vert Z\vert\psi\rangle$.

In [ ]:
theta = np.pi / 8
psi = ...  # YOUR CODE HERE

ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)

p0 = ...  # YOUR CODE HERE
p1 = ...  # YOUR CODE HERE

Z = np.array([[1, 0], [0, -1]], dtype=complex)

ex_z_from_probs = ...  # YOUR CODE HERE
ex_z_direct = ...  # YOUR CODE HERE

print(f"P(0) = {p0:.6f}")
print(f"P(1) = {p1:.6f}")
print(f"P(0)+P(1) = {p0+p1:.6f}")
print(f"<Z> from probs:   {np.real(ex_z_from_probs):.6f}")
print(f"<Z> as <psi|Z|psi>: {np.real(ex_z_direct):.6f}")

---
## Part 1: Math tasks

### M3.1. Single-qubit expectation values

For
$\lvert\psi\rangle = \cos(\pi/8)\lvert0\rangle + \sin(\pi/8)\lvert1\rangle$,
compute $\langle Z\rangle$, $\langle X\rangle$, and $\langle Y\rangle$.

In [ ]:
theta = np.pi / 8
psi = ...  # YOUR CODE HERE

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

ex_z = ...  # YOUR CODE HERE
ex_x = ...  # YOUR CODE HERE
ex_y = ...  # YOUR CODE HERE

print(f"<Z> = {np.real(ex_z):.6f}")
print(f"<X> = {np.real(ex_x):.6f}")
print(f"<Y> = {np.real(ex_y):.6f}")

### M3.2. Bell-state observables

For $\lvert\Phi^+\rangle = (\lvert00\rangle + \lvert11\rangle)/\sqrt{2}$,
compute $\langle Z\otimes Z\rangle$, $\langle Z\otimes I\rangle$, and $\langle X\otimes X\rangle$.

In [ ]:
phi_plus = ...  # YOUR CODE HERE

I = np.eye(2, dtype=complex)
ZZ = np.kron(Z, Z)
ZI = np.kron(Z, I)
XX = np.kron(X, X)

ex_zz = ...  # YOUR CODE HERE
ex_zi = ...  # YOUR CODE HERE
ex_xx = ...  # YOUR CODE HERE

print(f"<ZZ> = {np.real(ex_zz):.6f}")
print(f"<ZI> = {np.real(ex_zi):.6f}")
print(f"<XX> = {np.real(ex_xx):.6f}")

### M3.3. Hoeffding bound and shot count

Use Hoeffding:
$N \ge \ln(2/\delta)/(2\epsilon^2)$,
for $\epsilon=0.01$, confidence $95\%$ ($\delta=0.05$).

In [ ]:
epsilon = 0.01
delta = 0.05
n_min = ...  # YOUR CODE HERE
print(f"Minimum N from Hoeffding bound: {int(np.ceil(n_min))}")

---
## Part 2: Programming tasks

### P3.1. Sampler vs Estimator on one circuit

Use `StatevectorSampler` for counts and `StatevectorEstimator` for expectation values.

In [ ]:
theta = 1.2

sampler = StatevectorSampler()
estimator = StatevectorEstimator()

# YOUR CODE HERE:
# 1. Build a measured circuit with ry(theta)
# 2. Run sampler and print counts
# 3. Build an unmeasured circuit with ry(theta)
# 4. Run estimator with observable Z and print <Z>

In [ ]:
# YOUR CODE HERE: plot counts histogram (if counts exists)

### P3.2. PennyLane expectation value curve

Create a one-qubit $R_y(\theta)$ circuit and plot $\langle Z\rangle$.

In [ ]:
dev = qml.device("default.qubit", wires=1)


@qml.qnode(dev)
def expval_z(theta):
    # YOUR CODE HERE
    ...

thetas = np.linspace(0, 2 * np.pi, 120)
vals = np.array([expval_z(t) for t in thetas], dtype=float)

# YOUR CODE HERE: plot vals and compare to cos(theta)

### P3.3. Shot-scaling experiment

Estimate $\langle Z\rangle$ from counts for shots in `[10, 100, 1000, 10000]` and compare with exact.

In [ ]:
theta = np.pi / 4
true_ev = np.cos(theta)

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

shots_values = [10, 100, 1000, 10000]
estimates = []

# YOUR CODE HERE:
# loop over shots_values, run sampler, estimate <Z> = p0 - p1, and plot

### P3.4. Bell correlation check in Qiskit

Verify strong correlation on $\langle Z_0 Z_1\rangle$ for a Bell state.

In [ ]:
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)

ev_zz = ...  # YOUR CODE HERE
ev_zi = ...  # YOUR CODE HERE

print(f"<ZZ> = {float(np.real(ev_zz)):+.6f}")
print(f"<ZI> = {float(np.real(ev_zi)):+.6f}")

### P3.5. Real dataset mini task: Iris with quantum measurement features

Use two Iris classes (versicolor/virginica) and compare:

1. logistic regression on standardized raw features
2. logistic regression on quantum-derived features
$\left(\langle X\rangle,\langle Y\rangle,\langle Z\rangle\right)$ from one-qubit encoding.

In [ ]:
# Suggested scaffold
iris = load_iris()
X_all = iris.data[:, 2:4]
y_all = iris.target

# YOUR CODE HERE:
# 1) keep classes 1 and 2
# 2) train/test split + standardize
# 3) map to angles in [0, pi]
# 4) define a one-qubit qnode returning (<X>, <Y>, <Z>)
# 5) build quantum feature matrix
# 6) compare logistic regression accuracies (raw vs quantum-derived)

---
## Summary

Shot-based measurement estimates probabilities and expectation values with statistical uncertainty that decreases as $1/\sqrt{N}$. Observables specify what quantity is extracted from a state, and Qiskit's sampler/estimator primitives separate distribution questions from expectation-value questions.

The Iris mini task demonstrates an early real-data bridge: quantum-style measurement features can be used directly in a classical classifier.

---
## Optional extension: fidelity / SWAP test (preview)

For two pure states, fidelity is
\[
F(\psi,\varphi)=\lvert\langle\psi\vert\varphi\rangle\rvert^2.
\]
A SWAP-test style estimate uses the ancilla relation
\[
P(\mathrm{anc}=0)=\frac{1+F}{2}.
\]

Implement a minimal numerical example using this relation.

In [ ]:
# Optional preview scaffold:
# implement a small SWAP-test style routine and compare an estimated
# fidelity with an exact statevector overlap.


---
## Optional extension: mixed states and depolarizing noise

For a mixed state (density matrix) $\rho$:

- $P(0)=\rho_{00}$ and $P(1)=\rho_{11}$ for a $Z$-basis measurement.
- Expectation values are computed as $\langle O\rangle=\mathrm{Tr}(\rho O)$.

A simple one-qubit depolarizing channel is
$\rho'=(1-p)\rho + p I/2$.
Show numerically that for Pauli $Z$, $\langle Z\rangle$ shrinks by a factor $(1-p)$.

In [ ]:
# Optional preview scaffold:
# build rho = |psi><psi|, apply a depolarizing channel,
# and verify how <Z> changes as p increases.
